In [2]:
# IPL Machine Learning Models - Complete Training Pipeline
# This notebook trains 3 ML models for IPL prediction:
# 1. Match Winner Prediction (Random Forest)
# 2. Batsman Performance Prediction (Statistical Model)
# 3. Live Win Probability (LSTM Neural Network)

# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================
import json
import pandas as pd
import glob
import os
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump

warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")

# ============================================================
# CELL 2: SET DATA PATH AND LOAD FILES
# ============================================================
# Update this path to your IPL JSON dataset folder
folder_path = r"C:\Users\VASU MONPARA\OneDrive\Desktop\Cricket\Dataset\ipl_json"

files = glob.glob(folder_path + "/*.json")
print(f"📁 Found {len(files)} IPL match files")

if len(files) == 0:
    print("❌ No files found! Please check the folder path.")
else:
    print(f"✅ Ready to process {len(files)} matches")

# ============================================================
# CELL 3: EXTRACT DATA FROM JSON FILES
# ============================================================
print("🔄 Starting data extraction for all 3 tasks...\n")

match_list = []
ball_list = []
player_list = []

for file in tqdm(files, desc="Processing matches"):
    with open(file, encoding='utf-8') as f:
        try:
            data = json.load(f)
        except:
            continue
    
    match_id = os.path.basename(file).replace('.json', '')
    info = data.get('info', {})
    outcome = info.get('outcome', {})
    
    # Skip matches with D/L method or no clear winner
    if outcome.get('method') == 'D/L' or 'winner' not in outcome:
        continue
    
    winner = outcome['winner']
    
    # Skip incomplete matches
    if len(data.get('innings', [])) < 2:
        continue
    
    teams = info.get('teams', [])
    if len(teams) < 2:
        continue
        
    team1, team2 = teams[0], teams[1]
    
    # ========================================
    # TASK 1: MATCH WINNER DATA
    # ========================================
    toss = info.get('toss', {})
    match_list.append({
        'match_id': match_id,
        'season': info.get('season', 'Unknown'),
        'team1': team1,
        'team2': team2,
        'toss_winner': toss.get('winner', team1),
        'toss_decision': toss.get('decision', 'bat'),
        'winner': winner,
        'venue': info.get('venue', 'Unknown')
    })
    
    # ========================================
    # TASK 2: PLAYER RUNS DATA (BOTH INNINGS)
    # ========================================
    for inning in data.get('innings', []):
        batting_team = inning.get('team', 'Unknown')
        for over in inning.get('overs', []):
            for delivery in over.get('deliveries', []):
                batter = delivery.get('batter', 'Unknown')
                runs = delivery.get('runs', {}).get('batter', 0)
                player_list.append({
                    'match_id': match_id,
                    'batter': batter,
                    'team': batting_team,
                    'runs': runs
                })
    
    # ========================================
    # TASK 3: WIN PROBABILITY DATA (2ND INNINGS)
    # ========================================
    try:
        # Calculate first innings total
        first_innings = data['innings'][0]
        first_total = sum(
            d.get('runs', {}).get('total', 0) 
            for o in first_innings.get('overs', []) 
            for d in o.get('deliveries', [])
        )
        
        target = first_total + 1
        chasing_team = data['innings'][1].get('team', 'Unknown')
        
        # Track second innings ball-by-ball
        runs_scored = 0
        wickets_fallen = 0
        legal_balls = 0
        
        for over in data['innings'][1].get('overs', []):
            for delivery in over.get('deliveries', []):
                extras_dict = delivery.get('extras', {})
                is_wide = 'wides' in extras_dict
                is_noball = 'noballs' in extras_dict
                is_legal = not (is_wide or is_noball)
                
                # Update runs
                runs_scored += delivery.get('runs', {}).get('total', 0)
                
                # Update wickets (excluding run outs on wides/noballs)
                if 'wickets' in delivery:
                    for wicket in delivery['wickets']:
                        if not (is_wide or is_noball) or wicket.get('kind') == 'run out':
                            wickets_fallen += 1
                
                # Only increment ball count for legal deliveries
                if is_legal:
                    legal_balls += 1
                
                # Store state after each delivery
                ball_list.append({
                    'match_id': match_id,
                    'legal_ball': legal_balls,
                    'runs_so_far': runs_scored,
                    'wickets_fallen': wickets_fallen,
                    'runs_needed': max(target - runs_scored, 0),
                    'balls_remaining': max(120 - legal_balls, 0),
                    'wickets_remaining': max(10 - wickets_fallen, 0),
                    'win': 1 if winner == chasing_team else 0
                })
    except Exception as e:
        continue

print(f"\n✅ Data extraction complete!")
print(f"📊 Total matches processed: {len(match_list)}")
print(f"🏏 Total deliveries tracked: {len(ball_list)}")
print(f"👤 Total player records: {len(player_list)}")

# ============================================================
# CELL 4: CREATE AND SAVE DATASETS
# ============================================================
print("\n💾 Creating DataFrames and saving to CSV...\n")

# Create DataFrames
matches_df = pd.DataFrame(match_list)
balls_df = pd.DataFrame(ball_list)
players_df = pd.DataFrame(player_list)

# Display basic info
print("Match Data Shape:", matches_df.shape)
print("Ball Data Shape:", balls_df.shape)
print("Player Data Shape:", players_df.shape)
print()

# Save to CSV
matches_df.to_csv("matches_for_task1.csv", index=False)
balls_df.to_csv("balls_for_task3.csv", index=False)
players_df.to_csv("players_for_task2.csv", index=False)

print("✅ All datasets saved successfully!")
print("\nFiles created:")
print("  • matches_for_task1.csv")
print("  • balls_for_task3.csv")
print("  • players_for_task2.csv")

# Display sample data
print("\n📋 Sample Match Data:")
print(matches_df.head(3))

# ============================================================
# CELL 5: TASK 1 - MATCH WINNER PREDICTION MODEL
# ============================================================
print("\n" + "="*60)
print("TASK 1: MATCH WINNER PREDICTION (RANDOM FOREST)")
print("="*60 + "\n")

# Load data
df1 = pd.read_csv("matches_for_task1.csv")

# Encode teams
le_team = LabelEncoder()
all_teams = pd.unique(df1[['team1', 'team2']].values.ravel())
le_team.fit(all_teams)

print(f"Total unique teams: {len(all_teams)}")
print(f"Teams: {', '.join(all_teams)}\n")

# Create features
df1['t1'] = le_team.transform(df1['team1'])
df1['t2'] = le_team.transform(df1['team2'])
df1['toss'] = le_team.transform(df1['toss_winner'])
df1['team1_win'] = (df1['team1'] == df1['winner']).astype(int)

# Prepare features and target
X1 = df1[['t1', 't2', 'toss', 'toss_decision']]
X1 = pd.get_dummies(X1, columns=['toss_decision'], drop_first=False)
y1 = df1['team1_win']

print(f"Features shape: {X1.shape}")
print(f"Feature columns: {list(X1.columns)}\n")

# Split data
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train1)}")
print(f"Testing samples: {len(X_test1)}\n")

# Train Random Forest model
print("🔄 Training Random Forest model...")
model1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model1.fit(X_train1, y_train1)

# Evaluate
y_pred1 = model1.predict(X_test1)
acc1 = accuracy_score(y_test1, y_pred1)

print(f"\n✅ Task 1 Complete!")
print(f"📊 Match Winner Accuracy: {acc1*100:.2f}%")

# Detailed metrics
print("\n📈 Classification Report:")
print(classification_report(y_test1, y_pred1, target_names=['Team 2 Wins', 'Team 1 Wins']))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X1.columns,
    'importance': model1.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 Top Feature Importances:")
print(feature_importance.head())

# ============================================================
# CELL 6: TEST MATCH WINNER PREDICTION
# ============================================================
print("\n" + "="*60)
print("TESTING MATCH WINNER PREDICTION")
print("="*60 + "\n")

# Example prediction
team_a = "Mumbai Indians"
team_b = "Chennai Super Kings"
toss_winner_team = team_a
toss_dec = "bat"

print(f"Match: {team_a} vs {team_b}")
print(f"Toss: {toss_winner_team} wins and chooses to {toss_dec}\n")

# Encode
a = le_team.transform([team_a])[0]
b = le_team.transform([team_b])[0]
toss = le_team.transform([toss_winner_team])[0]

# Create input
input_data = pd.DataFrame(
    [[a, b, toss, toss_dec]], 
    columns=['t1', 't2', 'toss', 'toss_decision']
)
input_data = pd.get_dummies(input_data, columns=['toss_decision'])

# Align columns
for col in X1.columns:
    if col not in input_data.columns:
        input_data[col] = 0
input_data = input_data[X1.columns]

# Predict
prob = model1.predict_proba(input_data)[0]

print(f"🎯 Prediction Results:")
print(f"  {team_a} win probability: {prob[1]*100:.1f}%")
print(f"  {team_b} win probability: {prob[0]*100:.1f}%")
print(f"\n🏆 Predicted Winner: {team_a if prob[1] > prob[0] else team_b}")

# ============================================================
# CELL 7: TASK 2 - BATSMAN PERFORMANCE MODEL
# ============================================================
print("\n" + "="*60)
print("TASK 2: BATSMAN PERFORMANCE PREDICTION")
print("="*60 + "\n")

# Load player data
df2 = pd.read_csv("players_for_task2.csv")

# Calculate statistics per match
batsman_match_runs = df2.groupby(['match_id', 'batter'])['runs'].sum().reset_index()

# Calculate career statistics
batsman_stats = batsman_match_runs.groupby('batter')['runs'].agg([
    'mean', 'std', 'count', 'sum', 'min', 'max'
]).reset_index()

batsman_stats.columns = ['batter', 'avg_runs', 'std_runs', 'matches', 
                         'total_runs', 'min_score', 'max_score']
batsman_stats = batsman_stats.round(2)

# Filter batsmen with at least 10 matches
batsman_stats = batsman_stats[batsman_stats['matches'] >= 10].sort_values(
    'avg_runs', ascending=False
)

print(f"Total batsmen with 10+ matches: {len(batsman_stats)}\n")
print("📊 Top 10 Batsmen by Average:")
print(batsman_stats[['batter', 'avg_runs', 'matches', 'max_score']].head(10))

# ============================================================
# CELL 8: TASK 3 - WIN PROBABILITY MODEL (DATA PREP)
# ============================================================
print("\n" + "="*60)
print("TASK 3: WIN PROBABILITY PREDICTION (LSTM)")
print("="*60 + "\n")

# Load ball-by-ball data
df3 = pd.read_csv("balls_for_task3.csv")

print(f"Total ball records: {len(df3)}")
print(f"Total matches: {df3['match_id'].nunique()}\n")

# Define features
features = ['runs_needed', 'balls_remaining', 'wickets_remaining']

# Scale features
scaler = MinMaxScaler()
df3[features] = scaler.fit_transform(df3[features])

print("✅ Features scaled using MinMaxScaler")
print(f"Feature columns: {features}\n")

# Create sequences for LSTM
print("🔄 Creating sequences for LSTM...")

sequences = []
labels = []
max_len = 120  # Maximum balls in an innings

for mid in tqdm(df3['match_id'].unique(), desc="Processing matches"):
    mdf = df3[df3['match_id'] == mid].sort_values('legal_ball')
    
    # Get sequence of states
    seq = mdf[features].values
    
    # Pad if necessary
    if len(seq) < max_len:
        pad = np.repeat(seq[-1:], max_len - len(seq), axis=0)
        seq = np.vstack([seq, pad])
    
    sequences.append(seq[:max_len])
    labels.append(mdf['win'].iloc[0])

X3 = np.array(sequences)
y3 = np.array(labels)

print(f"\n✅ Sequences created!")
print(f"Input shape: {X3.shape} (matches, timesteps, features)")
print(f"Labels shape: {y3.shape}")
print(f"Win distribution: {np.bincount(y3)}\n")

# Split data
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

print(f"Training sequences: {X_train3.shape[0]}")
print(f"Testing sequences: {X_test3.shape[0]}")

# ============================================================
# CELL 9: TASK 3 - LSTM MODEL DEFINITION AND TRAINING
# ============================================================
print("\n" + "="*60)
print("BUILDING AND TRAINING LSTM MODEL")
print("="*60 + "\n")

# Define LSTM model
class WinLSTM(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Use only the last timestep output
        last_output = lstm_out[:, -1, :]
        out = self.fc(last_output)
        return self.sigmoid(out)

# Initialize model
model3 = WinLSTM(input_size=3, hidden_size=64, num_layers=1)
print(f"Model architecture:\n{model3}\n")

# Training setup
optimizer = torch.optim.Adam(model3.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_train3, dtype=torch.float32),
    torch.tensor(y_train3, dtype=torch.float32).unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_test3, dtype=torch.float32),
    torch.tensor(y_test3, dtype=torch.float32).unsqueeze(1)
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Training loop
print("🔄 Training LSTM model...\n")
num_epochs = 10
train_losses = []
test_losses = []

for epoch in range(num_epochs):
    # Training phase
    model3.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model3(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model3.eval()
    test_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            predictions = model3(batch_x)
            loss = criterion(predictions, batch_y)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {avg_train_loss:.4f} - "
          f"Val Loss: {avg_test_loss:.4f}")

print("\n✅ Training complete!")

# ============================================================
# CELL 10: EVALUATE WIN PROBABILITY MODEL
# ============================================================
print("\n" + "="*60)
print("EVALUATING WIN PROBABILITY MODEL")
print("="*60 + "\n")

# Evaluate on test set
model3.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        predictions = model3(batch_x)
        all_predictions.extend(predictions.numpy())
        all_labels.extend(batch_y.numpy())

all_predictions = np.array(all_predictions).flatten()
all_labels = np.array(all_labels).flatten()

# Calculate accuracy
predicted_classes = (all_predictions > 0.5).astype(int)
accuracy = (predicted_classes == all_labels).mean()

print(f"📊 Test Accuracy: {accuracy*100:.2f}%\n")

# Classification report
print("📈 Classification Report:")
print(classification_report(all_labels, predicted_classes, 
                          target_names=['Loss', 'Win']))

# ============================================================
# CELL 11: TEST WIN PROBABILITY PREDICTION
# ============================================================
print("\n" + "="*60)
print("TESTING WIN PROBABILITY PREDICTION")
print("="*60 + "\n")

def predict_win_probability(runs_needed, balls_left, wickets_left):
    """Predict win probability for a given match state"""
    # Scale input
    state = scaler.transform([[runs_needed, balls_left, wickets_left]])
    
    # Create sequence
    seq = np.repeat(state, 120, axis=0)
    seq_tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0)
    
    # Predict
    model3.eval()
    with torch.no_grad():
        prob = model3(seq_tensor).item() * 100
    
    return round(prob, 2)

# Test scenarios
test_scenarios = [
    (50, 60, 8, "Comfortable chase"),
    (100, 60, 7, "Balanced situation"),
    (120, 30, 4, "Difficult chase"),
    (20, 36, 9, "Easy chase"),
    (80, 24, 3, "Very difficult")
]

print("🎯 Testing different match scenarios:\n")
for runs, balls, wickets, description in test_scenarios:
    prob = predict_win_probability(runs, balls, wickets)
    rrr = (runs / balls) * 6 if balls > 0 else 0
    
    print(f"{description}:")
    print(f"  Runs: {runs}, Balls: {balls}, Wickets: {wickets}")
    print(f"  Required RRR: {rrr:.2f}")
    print(f"  Win Probability: {prob}%\n")

# ============================================================
# CELL 12: SAVE ALL MODELS
# ============================================================
print("\n" + "="*60)
print("SAVING ALL MODELS")
print("="*60 + "\n")

# Create directory
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

# Save Task 1: Random Forest model
dump(model1, os.path.join(save_dir, "random_forest_task1.pkl"))
print("✅ Saved: random_forest_task1.pkl")

# Save Task 3: Scaler
dump(scaler, os.path.join(save_dir, "task3_scaler.pkl"))
print("✅ Saved: task3_scaler.pkl")

# Save Task 3: LSTM model
torch.save(model3.state_dict(), os.path.join(save_dir, "winlstm_task3_state.pt"))
print("✅ Saved: winlstm_task3_state.pt")

torch.save(model3, os.path.join(save_dir, "winlstm_task3_full.pt"))
print("✅ Saved: winlstm_task3_full.pt")

print(f"\n🎉 All models saved successfully in '{save_dir}/' directory!")
print("\n📦 Files created:")
print("  1. random_forest_task1.pkl - Match winner model")
print("  2. task3_scaler.pkl - Feature scaler for LSTM")
print("  3. winlstm_task3_state.pt - LSTM state dict")
print("  4. winlstm_task3_full.pt - Complete LSTM model")

print("\n" + "="*60)
print("MODEL TRAINING COMPLETE! 🎉")
print("="*60)
print("\n✅ Ready to run the Streamlit app!")
print("Run: streamlit run cricket_ml_app.py")

✅ All libraries imported successfully!
PyTorch version: 2.9.1+cpu
Pandas version: 2.3.3
📁 Found 1169 IPL match files
✅ Ready to process 1169 matches
🔄 Starting data extraction for all 3 tasks...



Processing matches: 100%|██████████| 1169/1169 [00:33<00:00, 34.92it/s]



✅ Data extraction complete!
📊 Total matches processed: 1124
🏏 Total deliveries tracked: 130530
👤 Total player records: 269613

💾 Creating DataFrames and saving to CSV...

Match Data Shape: (1124, 8)
Ball Data Shape: (130530, 8)
Player Data Shape: (269613, 4)

✅ All datasets saved successfully!

Files created:
  • matches_for_task1.csv
  • balls_for_task3.csv
  • players_for_task2.csv

📋 Sample Match Data:
  match_id season                   team1                        team2  \
0  1082591   2017     Sunrisers Hyderabad  Royal Challengers Bangalore   
1  1082592   2017  Rising Pune Supergiant               Mumbai Indians   
2  1082593   2017           Gujarat Lions        Kolkata Knight Riders   

                   toss_winner toss_decision                  winner  \
0  Royal Challengers Bangalore         field     Sunrisers Hyderabad   
1       Rising Pune Supergiant         field  Rising Pune Supergiant   
2        Kolkata Knight Riders         field   Kolkata Knight Riders   

    

Processing matches: 100%|██████████| 1124/1124 [00:00<00:00, 1548.02it/s]



✅ Sequences created!
Input shape: (1124, 120, 3) (matches, timesteps, features)
Labels shape: (1124,)
Win distribution: [519 605]

Training sequences: 899
Testing sequences: 225

BUILDING AND TRAINING LSTM MODEL

Model architecture:
WinLSTM(
  (lstm): LSTM(3, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

🔄 Training LSTM model...

Epoch 1/10 - Train Loss: 0.6681 - Val Loss: 0.6170
Epoch 2/10 - Train Loss: 0.5527 - Val Loss: 0.4414
Epoch 3/10 - Train Loss: 0.4205 - Val Loss: 0.3807
Epoch 4/10 - Train Loss: 0.4009 - Val Loss: 0.4125
Epoch 5/10 - Train Loss: 0.4172 - Val Loss: 0.3850
Epoch 6/10 - Train Loss: 0.3601 - Val Loss: 0.3436
Epoch 7/10 - Train Loss: 0.3360 - Val Loss: 0.3875
Epoch 8/10 - Train Loss: 0.3431 - Val Loss: 0.3386
Epoch 9/10 - Train Loss: 0.3297 - Val Loss: 0.3155
Epoch 10/10 - Train Loss: 0.2991 - Val Loss: 0.3124

✅ Training complete!

EVALUATING WIN PROBABILITY MODEL

📊 Test Accuracy: 86.22%

📈 Classificati